# 30. Swin Transformer 계층적 비전 모델

이 노트북은 `29_DETR_객체_탐지_Transformer.ipynb` 다음 단계로, Swin Transformer가 Transformer를 비전 backbone으로 쓰기 위해 어떤 구조를 도입했는지 이해합니다.

기본 ViT는 전체 patch token 사이 self-attention을 계산합니다. 이미지가 커지면 token 수가 많아져 계산량이 커집니다. Swin Transformer는 local window attention, shifted window, patch merging을 사용해 효율적인 계층적 feature를 만듭니다.

이번 노트북의 목표는 다음과 같습니다.

- window attention이 전체 attention보다 효율적인 이유를 이해합니다.
- shifted window가 window 사이 정보 교환을 돕는 방식을 확인합니다.
- patch merging으로 계층적 feature map을 만드는 과정을 이해합니다.
- Swin이 classification, detection, segmentation backbone으로 쓰이기 좋은 이유를 정리합니다.

## 30-1. 준비

작은 token grid를 사용해 Swin Transformer의 핵심 아이디어를 시각화합니다. 실제 모델 구현보다 window partition, shift, patch merging의 모양을 이해하는 데 초점을 둡니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)

## 30-2. 전체 attention의 계산 부담

self-attention은 token 수가 `N`일 때 attention matrix 크기가 `N x N`입니다. 이미지 token이 많아질수록 계산량이 빠르게 커집니다.

Swin은 전체 token을 한 번에 모두 보지 않고, 작은 window 안에서만 attention을 계산합니다.

In [ ]:
sizes = np.array([14, 28, 56])
tokens = sizes ** 2
global_attention_pairs = tokens ** 2
window_size = 7
window_attention_pairs = tokens * (window_size ** 2)

for s, n, g, w in zip(sizes, tokens, global_attention_pairs, window_attention_pairs):
    print(f'{s}x{s} tokens: global={g:,}, window={w:,}')

plt.figure(figsize=(6, 3.5))
plt.plot(tokens, global_attention_pairs, marker='o', label='global attention')
plt.plot(tokens, window_attention_pairs, marker='o', label='window attention')
plt.xlabel('number of tokens')
plt.ylabel('attention pair count')
plt.title('Global attention vs window attention')
plt.legend()
plt.show()

## 30-3. Window partition

Swin은 feature map을 격자로 유지한 채, 가까운 patch token들을 window로 묶습니다. 같은 window 안의 token끼리만 attention을 계산하므로 계산량이 줄어듭니다.

In [ ]:
grid_size = 8
window = 4
grid = np.arange(grid_size * grid_size).reshape(grid_size, grid_size)

def draw_window_grid(ax, data, title, window_size=4):
    ax.imshow(data, cmap='Blues')
    ax.set_title(title)
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(color='white', linewidth=0.6)
    for y in range(0, grid_size, window_size):
        for x in range(0, grid_size, window_size):
            rect = Rectangle((x - 0.5, y - 0.5), window_size, window_size, fill=False, edgecolor='#dc2626', linewidth=2.5)
            ax.add_patch(rect)

fig, ax = plt.subplots(figsize=(5, 5))
draw_window_grid(ax, grid, '4 x 4 window partition')
plt.show()

In [ ]:
def partition_windows(data, window_size):
    windows = []
    h, w = data.shape
    for y in range(0, h, window_size):
        for x in range(0, w, window_size):
            windows.append(data[y:y + window_size, x:x + window_size])
    return windows

windows = partition_windows(grid, window)
for i, win in enumerate(windows):
    print(f'window {i}: tokens {win.reshape(-1).tolist()}')

window attention만 반복하면 서로 다른 window에 속한 token끼리 직접 정보를 주고받기 어렵습니다. 그래서 Swin은 다음 block에서 window를 반 칸 정도 이동한 shifted window를 사용합니다.

## 30-4. Shifted window

shifted window는 window 경계를 바꿔 이전 block에서 다른 window였던 token들이 다음 block에서는 같은 window에 들어오게 합니다. 이렇게 local attention을 유지하면서도 window 사이 정보가 섞입니다.

In [ ]:
shift = window // 2
shifted = np.roll(grid, shift=(-shift, -shift), axis=(0, 1))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
draw_window_grid(axes[0], grid, 'regular windows', window)
draw_window_grid(axes[1], shifted, 'shifted token grid', window)
plt.tight_layout()
plt.show()

실제 Swin 구현에서는 cyclic shift와 attention mask를 함께 사용해 경계가 잘못 섞이지 않도록 처리합니다. 개념적으로는 window 경계를 바꾸어 이웃 window 사이 정보 교환을 가능하게 만드는 것이 핵심입니다.

## 30-5. Patch merging으로 계층 구조 만들기

CNN backbone은 layer가 깊어질수록 공간 해상도는 줄고 channel 수는 늘어나는 계층적 feature map을 만듭니다. Swin도 patch merging으로 비슷한 구조를 만듭니다.

```text
H x W x C -> H/2 x W/2 x 4C -> linear projection -> H/2 x W/2 x 2C
```

이렇게 하면 detection이나 segmentation에서 필요한 multi-scale feature를 만들기 좋습니다.

In [ ]:
feature = np.arange(8 * 8 * 2).reshape(8, 8, 2)

def patch_merge(x):
    h, w, c = x.shape
    merged = []
    for y in range(0, h, 2):
        row = []
        for x_pos in range(0, w, 2):
            block = x[y:y + 2, x_pos:x_pos + 2, :].reshape(-1)
            row.append(block)
        merged.append(row)
    return np.array(merged)

merged = patch_merge(feature)
print('before patch merging:', feature.shape)
print('after  patch merging:', merged.shape)
print('one merged token contains:', merged[0, 0])

In [ ]:
stage_shapes = [
    ('Stage 1', 56, 56, 96),
    ('Stage 2', 28, 28, 192),
    ('Stage 3', 14, 14, 384),
    ('Stage 4', 7, 7, 768),
]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('off')
ax.set_title('Swin의 계층적 feature map 예시')

for i, (name, h, w, c) in enumerate(stage_shapes):
    x = i * 2.0
    size = 1.4 / (i + 1) + 0.25
    rect = Rectangle((x, 1 - size / 2), size, size, facecolor='#dbeafe', edgecolor='#2563eb', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + size / 2, 1, f'{h}x{w}\nC={c}', ha='center', va='center', fontsize=9)
    ax.text(x + size / 2, 0.05, name, ha='center', va='center')
    if i < len(stage_shapes) - 1:
        ax.annotate('', xy=(x + 1.75, 1), xytext=(x + size + 0.1, 1), arrowprops=dict(arrowstyle='->'))

ax.set_xlim(-0.2, 7.4)
ax.set_ylim(-0.2, 2.0)
plt.show()

## 30-6. Swin이 backbone으로 쓰이기 좋은 이유

| 구성 | 역할 |
|---|---|
| window attention | attention 계산량을 줄이고 local pattern을 효율적으로 봅니다. |
| shifted window | window 사이 정보가 흐르도록 만듭니다. |
| patch merging | CNN처럼 계층적 feature map을 만듭니다. |
| multi-stage feature | classification뿐 아니라 detection, segmentation에도 적합합니다. |

Swin은 Transformer 기반 모델이면서도 CNN backbone이 제공하던 multi-scale 구조를 잘 가져왔기 때문에, 이미지 분류뿐 아니라 Mask R-CNN, UperNet 같은 detection/segmentation 모델의 backbone으로도 널리 사용됩니다.

## 정리

- Swin Transformer는 전체 attention 대신 window attention을 사용해 계산량을 줄입니다.
- shifted window는 서로 다른 window 사이의 정보 교환을 가능하게 합니다.
- patch merging은 해상도를 줄이고 channel 정보를 늘려 계층적 feature map을 만듭니다.
- Swin은 Transformer 기반이면서 CNN backbone처럼 다양한 비전 task에 연결하기 좋습니다.

다음 노트북 `31_SAM_Segment_Anything_개념.ipynb`에서는 promptable segmentation과 foundation vision model의 흐름으로 넘어갑니다.